# HGSplat — Weather-aware Heatmap Loss for LongSplat

**이윤호 / KNUVI**

| 섹션 | 내용 |
|------|------|
| 0 | GPU / CUDA 환경 확인 |
| 1 | 레포 클론 및 의존성 설치 |
| 2 | Google Drive 마운트 및 경로 설정 |
| 3 | Heatmap 생성 (MWFormer) |
| 4 | 학습 — Baseline |
| 5 | 학습 — Ours (Heatmap Loss) |
| 6 | 결과 비교 |

> **런타임 설정:** 런타임 → 런타임 유형 변경 → **L4 GPU** 선택

---
## Section 0 — GPU / CUDA 환경 확인

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

---
## Section 1 — 레포 클론 및 의존성 설치

In [ ]:
!git clone --recursive https://github.com/Leeyoonho02/HGSplat.git
%cd HGSplat

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ.get('PATH', '')
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

!pip install -q submodules/simple-knn --no-build-isolation
!pip install -q submodules/diff-gaussian-rasterization --no-build-isolation
!pip install -q submodules/fused-ssim --no-build-isolation

In [ ]:
# (선택) RoPE cuda 커널 컴파일 — 속도 향상, 약 2분 소요
!cd submodules/mast3r/dust3r/croco/models/curope && python setup.py build_ext --inplace
%cd /content/HGSplat

---
## Section 2 — Google Drive 마운트 및 경로 설정

Drive 구조 (예시):
```
MyDrive/HGSplat/
├── scenes/
│   └── snow_scene/
│       ├── images/        ← 입력 프레임
│       └── heatmaps/      ← Section 3에서 생성 (또는 사전 업로드)
└── checkpoints/
    ├── backbone.pth       ← MWFormer Network_top 체크포인트
    └── style_filter.pth   ← MWFormer StyleFilter 체크포인트
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ★ 경로 설정 — 여기만 수정
DRIVE_ROOT   = '/content/drive/MyDrive/HGSplat'
SCENE_NAME   = 'snow_scene'
SCENE_DIR    = f'{DRIVE_ROOT}/scenes/{SCENE_NAME}'
CKPT_BACKBONE = f'{DRIVE_ROOT}/checkpoints/backbone.pth'      # Network_top
CKPT_STYLE    = f'{DRIVE_ROOT}/checkpoints/style_filter.pth'  # StyleFilter

!mkdir -p data
!ln -sfn {SCENE_DIR} data/{SCENE_NAME}
!ls data/{SCENE_NAME}/

---
## Section 3 — Heatmap 생성 (MWFormer)

> **사전 생성 옵션:** 연구실 서버에서 미리 생성한 뒤 Drive의 `heatmaps/` 폴더에 올려두면 이 섹션 건너뜀.

MWFormer 구조:
- `StyleFilter` : 입력 이미지 → 날씨 스타일 벡터
- `Network_top` : (이미지, 스타일 벡터) → 복원 이미지
- Heatmap = `|입력 - 복원|` → weight map `W_t = exp(-α·H_t)`

In [ ]:
# heatmaps/ 폴더 이미 있으면 건너뛰기
import os
heatmap_dir = f'data/{SCENE_NAME}/heatmaps'

if os.path.isdir(heatmap_dir) and len(os.listdir(heatmap_dir)) > 0:
    print(f'[skip] {heatmap_dir} 이미 존재 ({len(os.listdir(heatmap_dir))}개)')
else:
    print('[info] heatmap 생성 필요 — 아래 셀 실행')

In [ ]:
# MWFormer 레포 클론
%cd /content
!git clone https://github.com/taco-group/MWFormer.git
%cd /content/HGSplat

In [ ]:
# Heatmap 생성
# --alpha 는 학습 시 --heatmap_alpha 와 동일한 값 사용
!python generate_heatmaps.py \
    --mwformer_root /content/MWFormer \
    --ckpt_backbone {CKPT_BACKBONE} \
    --ckpt_style    {CKPT_STYLE} \
    --scene_dir     data/{SCENE_NAME}/images \
    --out_dir       data/{SCENE_NAME}/heatmaps \
    --alpha         5.0

print('생성된 heatmap 수:', len(os.listdir(f'data/{SCENE_NAME}/heatmaps')))

---
## Section 4 — 학습: Baseline

`heatmaps/` 폴더가 없으면 자동으로 일반 L1 loss (Baseline) 로 동작

In [ ]:
BASELINE_OUT = 'output/baseline'

# heatmaps/ 폴더를 임시로 숨겨서 Baseline 강제
!mv data/{SCENE_NAME}/heatmaps data/{SCENE_NAME}/heatmaps_bak 2>/dev/null; true

!python train.py \
    -s data/{SCENE_NAME} \
    -m {BASELINE_OUT} \
    --mode custom \
    --resolution 2

# heatmaps/ 복구
!mv data/{SCENE_NAME}/heatmaps_bak data/{SCENE_NAME}/heatmaps 2>/dev/null; true

!python render.py  -m {BASELINE_OUT}
!python metrics.py -m {BASELINE_OUT}

---
## Section 5 — 학습: Ours (Heatmap Loss)

`data/{SCENE_NAME}/heatmaps/` 폴더가 존재하면 자동 활성화

In [ ]:
ALPHA = 5.0
OURS_OUT = f'output/ours_alpha{ALPHA}'

!python train.py \
    -s data/{SCENE_NAME} \
    -m {OURS_OUT} \
    --mode custom \
    --resolution 2 \
    --heatmap_alpha {ALPHA}

!python render.py  -m {OURS_OUT}
!python metrics.py -m {OURS_OUT}

---
## Section 6 — 결과 비교

In [ ]:
import json, glob
import pandas as pd

def load_metrics(output_dir):
    paths = glob.glob(f'{output_dir}/**/results.json', recursive=True)
    if not paths:
        return None
    with open(paths[0]) as f:
        data = json.load(f)
    return list(data.values())[-1]

rows = []
for name, out in [('Baseline', BASELINE_OUT), (f'Ours (α={ALPHA})', OURS_OUT)]:
    m = load_metrics(out)
    rows.append({'Model': name, **(m if m else {'PSNR': '-', 'SSIM': '-', 'LPIPS': '-'})})

print(pd.DataFrame(rows).set_index('Model').to_string())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def get_render_sample(output_dir):
    imgs = sorted(glob.glob(f'{output_dir}/**/renders/*.png', recursive=True))
    return imgs[0] if imgs else None

baseline_img = get_render_sample(BASELINE_OUT)
ours_img     = get_render_sample(OURS_OUT)

if baseline_img and ours_img:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(mpimg.imread(baseline_img)); axes[0].set_title('Baseline');         axes[0].axis('off')
    axes[1].imshow(mpimg.imread(ours_img));     axes[1].set_title(f'Ours (α={ALPHA})'); axes[1].axis('off')
    plt.tight_layout()
    plt.savefig('comparison.png', dpi=150)
    plt.show()
else:
    print('렌더링 결과 없음 — 학습 먼저 실행하세요.')

In [ ]:
# 결과 Drive에 저장
import shutil
save_root = f'{DRIVE_ROOT}/results/{SCENE_NAME}'
os.makedirs(save_root, exist_ok=True)

for name, out in [('baseline', BASELINE_OUT), (f'ours_alpha{ALPHA}', OURS_OUT)]:
    if os.path.exists(out):
        shutil.copytree(out, f'{save_root}/{name}', dirs_exist_ok=True)
        print(f'[saved] {save_root}/{name}')

if os.path.exists('comparison.png'):
    shutil.copy('comparison.png', f'{save_root}/comparison.png')
    print(f'[saved] {save_root}/comparison.png')